# Connect to Drive

In [1]:
from src.utils.colab import is_environment_colab_notebook

if is_environment_colab_notebook():
    from google.colab import drive
    drive.mount('/content/drive')

In [2]:
import os

project_folder = "./"
data_folder = os.path.join(project_folder, "data")
world_model_folder = os.path.join(project_folder, "weights/worldmodel")
output_folder = os.path.join(project_folder, "weights/controller")
os.makedirs(output_folder, exist_ok=True)

# Training Settings

In [ ]:
NUM_GENERATIONS = 500
POPULATION_SIZE = 64
STEPS_PER_ROLLOUT = 5000
ROLLOUTS_PER_SOLUTION = 16
SIGMA_INIT = 0.1

OBSERVATION_REPRESENTATION_DIM = 32
HIDDEN_DIM = 256
ACTION_DIM = 3

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "controller"

LOG_LEVEL = "INFO"

# Setup

In [4]:
from src.models.controller import Controller
from src.models.simulation_worldmodel import SimulationWorldModel
from src.training.controller import ControllerEvolutionaryTrainer
from src.utils.torch import get_device
from src.utils.logging import get_logger
from src.utils.secrets import get_secret

In [5]:
logger = get_logger()

2025-12-09 16:53:59 [INFO] Logger initialized.


In [6]:
DEVICE = get_device()

2025-12-09 16:53:59 [INFO] Logger initialized.
2025-12-09 16:53:59 [INFO] Using device: mps:0


# Train

In [7]:
world_model_path = os.path.join(world_model_folder, f"model.pth")
simulation_worldmodel = SimulationWorldModel(worldmodel_path=world_model_path,
                                             batch_size=ROLLOUTS_PER_SOLUTION,
                                             device=DEVICE,
                                             logger=logger)

In [8]:
model = Controller(observation_dim=OBSERVATION_REPRESENTATION_DIM,
                   hidden_dim=HIDDEN_DIM,
                   action_dim=ACTION_DIM,
                   device=DEVICE)

In [9]:
wandb_setup = {
    "api_key": get_secret('wandbApiKey'),
    "project": WANDB_PROJECT,
    "run_name": WANDB_RUN_NAME,
    "config": {
        "num_generations": NUM_GENERATIONS,
        "population_size": POPULATION_SIZE,
        "steps_per_rollout": STEPS_PER_ROLLOUT,
        "rollouts_per_solution": ROLLOUTS_PER_SOLUTION,
        "sigma_init": SIGMA_INIT,
        "hidden_dim": HIDDEN_DIM,
        "observation_representation_dim": OBSERVATION_REPRESENTATION_DIM,
        "architecture": "FEED-FORWARD"
    }
}

In [10]:
trainer = ControllerEvolutionaryTrainer(model=model,
                                        weights_folder=output_folder,
                                        num_generations=NUM_GENERATIONS,
                                        population_size=POPULATION_SIZE,
                                        simulation_world_model=simulation_worldmodel,
                                        steps_per_rollout=STEPS_PER_ROLLOUT,
                                        rollouts_per_solution=ROLLOUTS_PER_SOLUTION,
                                        sigma_init=SIGMA_INIT,
                                        load_checkpoint=True,
                                        device=DEVICE,
                                        wandb_setup=wandb_setup,
                                        logger=logger)
                                        

2025-12-09 16:53:59 [INFO] No existing checkpoints found. Starting from scratch.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/henriqueschmitz/.netrc
wandb: Currently logged in as: schhenrique (schhenrique-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


(32_w,64)-aCMA-ES (mu_w=17.6,w_1=11%) in dimension 867 (seed=375833, Tue Dec  9 16:54:01 2025)


In [ ]:
trainer.train()

Epoch:   0%|          | 0/100 [00:00<?, ?epoch/s]

Train Epoch 1:   0%|          | 0/64 [00:00<?, ?batch/s]

2025-12-09 16:55:51 [INFO] Epoch 1 Training Average Reward: -1229.4305
2025-12-09 16:55:51 [INFO] Epoch 1 Training Max Reward: -500.5154
